In [1]:
!pip install chromadb sentence-transformers python-dotenv rich -q

In [2]:
import re
from pathlib import Path
from typing import List, Dict
import chromadb
import sentence_transformers

print("Imports successful")

Imports successful


In [3]:
CORPUS_DIR = Path("corpus/zoning")

MAX_CHARS = 2400   # ~600 tokens approximation
MIN_CHARS = 50

In [4]:
def load_markdown_files(corpus_dir: Path) -> List[Dict]:

    documents = []

    for file_path in corpus_dir.glob("*.md"):

        with open(file_path, "r", encoding="utf-8") as f:

            lines = f.readlines()

            text = "".join(lines)


        documents.append({
            "source_file": file_path.name,
            "text": text,
            "lines": lines
        })

    return documents

In [5]:
def parse_header(text: str) -> Dict:

    lines = text.split("\n")

    document_title = ""
    last_amended = ""

    for line in lines[:10]:

        line = line.strip()

        # Extract title
        if line.startswith("# "):
            document_title = line.replace("# ", "").strip()

        # Extract amendment date
        if "**Last Amended:**" in line:
            last_amended = (
                line.replace("**Last Amended:**", "")
                .strip()
            )

    return {
        "document_title": document_title,
        "last_amended": last_amended
    }

In [6]:
def split_into_sections(
    lines: List[str]
) -> List[Dict]:

    sections = []

    current_section = None

    current_content = []

    start_line = None

    for idx, line in enumerate(lines, start=1):

        # ---------------------------------
        # Detect new section
        # ---------------------------------

        if line.strip().startswith("## "):

            # Save previous section
            if current_section is not None:

                sections.append({

                    "section_title": current_section,

                    "content": "".join(
                        current_content
                    ).strip(),

                    "start_line": start_line,

                    "end_line": idx - 1
                })

            # Start new section
            current_section = (
                line.strip()
                .replace("## ", "")
                .strip()
            )

            current_content = []

            start_line = idx

        else:

            # Accumulate section content
            if current_section is not None:

                current_content.append(line)

    # ---------------------------------
    # Save final section
    # ---------------------------------

    if current_section is not None:

        sections.append({

            "section_title": current_section,

            "content": "".join(
                current_content
            ).strip(),

            "start_line": start_line,

            "end_line": len(lines)
        })

    return sections

In [7]:
SUBSECTION_PATTERN = (
    r"^###\s+(\d{2}-\d{2,3})"
)


def split_into_subsections(
    lines: List[str]
) -> List[Dict]:

    subsections = []

    current_subsection = None

    current_content = []

    start_line = None

    for idx, line in enumerate(lines, start=1):

        stripped = line.strip()

        # ---------------------------------
        # Detect subsection header
        # ---------------------------------

        match = re.match(
            SUBSECTION_PATTERN,
            stripped
        )

        if match:

            # Save previous subsection
            if current_subsection is not None:

                subsections.append({

                    "section_id":
                    current_subsection,

                    "content":
                    "".join(
                        current_content
                    ).strip(),

                    "start_line":
                    start_line,

                    "end_line":
                    idx - 1
                })

            # ---------------------------------
            # Start new subsection
            # ---------------------------------

            current_subsection = (
                match.group(1)
            )

            current_content = [line]

            start_line = idx

        else:

            if current_subsection is not None:

                current_content.append(
                    line
                )

    # ---------------------------------
    # Save final subsection
    # ---------------------------------

    if current_subsection is not None:

        subsections.append({

            "section_id":
            current_subsection,

            "content":
            "".join(
                current_content
            ).strip(),

            "start_line":
            start_line,

            "end_line":
            len(lines)
        })

    return subsections

In [8]:
# def split_large_section(
#     text: str,
#     max_chars: int = MAX_CHARS
# ) -> List[str]:

#     if len(text) <= max_chars:
#         return [text]

#     paragraphs = text.split("\n\n")

#     chunks = []
#     current_chunk = ""

#     for para in paragraphs:

#         para = para.strip()

#         if not para:
#             continue

#         candidate = current_chunk + "\n\n" + para

#         if len(candidate) > max_chars:

#             if current_chunk.strip():
#                 chunks.append(current_chunk.strip())

#             current_chunk = para

#         else:
#             current_chunk = candidate

#     if current_chunk.strip():
#         chunks.append(current_chunk.strip())

#     return chunks

In [9]:
def split_large_section(
    text: str,
    max_chars: int = MAX_CHARS
) -> List[str]:

    if len(text) <= max_chars:
        return [text]

    subsection_pattern = r"(?=^### )"

    subsections = re.split(
        subsection_pattern,
        text,
        flags=re.MULTILINE
    )

    subsections = [
        s.strip()
        for s in subsections
        if s.strip()
    ]

    chunks = []

    current_chunk = ""

    for subsection in subsections:

        candidate = (
            current_chunk
            + "\n\n"
            + subsection
        )

        if len(candidate) > max_chars:

            if current_chunk.strip():
                chunks.append(
                    current_chunk.strip()
                )

            current_chunk = subsection

        else:
            current_chunk = candidate

    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks

In [10]:
def build_chunks(
    document: Dict
) -> List[Dict]:

    source_file = document["source_file"]

    text = document["text"]

    lines = document["lines"]

    header = parse_header(text)

    sections = split_into_subsections(
        lines
    )

    chunks = []

    chunk_index = 0

    for section in sections:

        section_id = section[
            "section_id"
        ]

        section_content = section[
            "content"
        ]

        start_line = section[
            "start_line"
        ]

        end_line = section[
            "end_line"
        ]

        split_chunks = split_large_section(
            section_content
        )

        total_chunks = len(split_chunks)

        total_lines = (
            end_line - start_line + 1
        )

        approx_lines_per_chunk = max(
            1,
            total_lines // total_chunks
        )

        for idx, chunk_text in enumerate(
            split_chunks
        ):

            chunk_text = chunk_text.strip()

            if len(chunk_text) < MIN_CHARS:
                continue

            # ---------------------------------
            # Extract cross references
            # ---------------------------------

            cross_refs = re.findall(

                r"\d{2}-\d{2,3}",

                chunk_text
            )

            cross_refs = list(set(cross_refs))

            # ---------------------------------
            # Remove self reference
            # ---------------------------------

            cross_refs = [

                ref

                for ref in cross_refs

                if ref != section_id
            ]

            # ---------------------------------
            # Serialize cross refs for Chroma
            # ---------------------------------

            cross_refs_str = (

                "|".join(cross_refs)

                if cross_refs

                else "NONE"
            )

            # ---------------------------------
            # Remove redundant markdown metadata
            # ---------------------------------

            chunk_text = re.sub(
                r"\*\*Source:\*\*.*?\n",
                "",
                chunk_text
            )

            chunk_text = re.sub(
                r"\*\*Last Amended:\*\*.*?\n",
                "",
                chunk_text
            )

            chunk_text = re.sub(
                r"\*\*Retrieved:\*\*.*?\n",
                "",
                chunk_text
            )

            # ---------------------------------
            # Approximate provenance lines
            # ---------------------------------

            chunk_start_line = (
                start_line
                + (idx * approx_lines_per_chunk)
            )

            chunk_end_line = min(

                end_line,

                chunk_start_line
                + approx_lines_per_chunk
                - 1
            )

            # ---------------------------------
            # Prepend visible metadata
            # ---------------------------------

            prepended_text = (

                f"[Source: {source_file} | "
                f"Section ID: {section_id} | "
                f"Amended: {header['last_amended']} | "
                f"Lines: {chunk_start_line}-{chunk_end_line}]"

                f"\n\n"

                f"{chunk_text}"
            )

            has_cross_ref = bool(
                cross_refs
            )

            chunks.append({

                "id":
                f"{source_file}_{chunk_index}",

                "document":
                prepended_text,

                "metadata": {

                    "source_file":
                    source_file,

                    "section_id":
                    section_id,

                    "cross_refs":
                    cross_refs_str,

                    "last_amended":
                    header["last_amended"],

                    "chunk_index":
                    chunk_index,

                    "has_cross_ref":
                    has_cross_ref,

                    "start_line":
                    chunk_start_line,

                    "end_line":
                    chunk_end_line
                }
            })

            chunk_index += 1

    return chunks

In [11]:
all_documents = load_markdown_files(
    CORPUS_DIR
)

all_chunks = []

file_stats = []

for document in all_documents:

    chunks = build_chunks(
        document
    )

    all_chunks.extend(
        chunks
    )

    header = parse_header(
        document["text"]
    )

    file_stats.append({

        "source_file":
        document["source_file"],

        "chunks_created":
        len(chunks),

        "last_amended":
        header["last_amended"]
    })

print(
    f"Total documents loaded: "
    f"{len(all_documents)}"
)

print(
    f"Total chunks created: "
    f"{len(all_chunks)}"
)


Total documents loaded: 10
Total chunks created: 8


In [12]:
all_chunks

[{'id': 'zr_03_rear_yard_requirements.md_0',
  'document': '[Source: zr_03_rear_yard_requirements.md | Section ID: 23-342 | Amended: 5/12/2021 | Lines: 9-31]\n\n### 23-342: Rear Yard Requirements\n\n**Applicable Districts:** R1 through R10\n\nIn all districts, as indicated, a **rear yard** shall be provided at every required **rear lot line** of a **zoning lot**, except as otherwise provided in Sections 23-341 (Permitted obstructions in required rear yards or rear yard equivalents) and 23-344 (Additional rear yard modifications).\n\nThe minimum required **rear yard** depth shall be:\n\n| District | Minimum Rear Yard Depth |\n|---|---|\n| R1 through R5 | 30 feet |\n| R6 through R10 | 30 feet |\n\nFor corner lots in any Residence District, no rear yard shall be required.\n\nFor through lots in any Residence District, each **street** frontage shall be subject to front yard requirements and no rear yard shall be required; however, where a through lot is also a corner lot, the lot shall be 

In [13]:
for chunk in all_chunks:

    print("=" * 80)

    print(chunk["id"])

    print(len(chunk["document"]))

zr_03_rear_yard_requirements.md_0
1476
zr_03_rear_yard_requirements.md_1
400
zr_03_rear_yard_requirements.md_2
761
zr_07_front_yard_requirements.md_0
1523
zr_07_front_yard_requirements.md_1
470
zr_10_height_setback_R6_R12.md_0
949
zr_10_height_setback_R6_R12.md_1
1026
zr_10_height_setback_R6_R12.md_2
629


In [14]:
import chromadb

from chromadb.utils.embedding_functions import (
    SentenceTransformerEmbeddingFunction
)

In [15]:
CHROMA_PATH = "chroma_db"

COLLECTION_NAME = "zoning_docs"

In [16]:
client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

In [17]:
embedding_function = (
    SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )
)

In [18]:
existing_collections = client.list_collections()

existing_names = [
    collection.name
    for collection in existing_collections
]

if COLLECTION_NAME in existing_names:

    client.delete_collection(
        name=COLLECTION_NAME
    )

    print(f"Deleted existing collection: {COLLECTION_NAME}")

Deleted existing collection: zoning_docs


In [19]:
collection = client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function
)

print("Collection created successfully")

Collection created successfully


In [20]:
chroma_documents = [

    chunk["document"]

    for chunk in all_chunks
]

metadatas = [

    chunk["metadata"]

    for chunk in all_chunks
]

ids = [

    chunk["id"]

    for chunk in all_chunks
]


In [21]:
collection.add(

    documents=chroma_documents,

    metadatas=metadatas,

    ids=ids
)

print(
    "Documents indexed successfully"
)

Documents indexed successfully


In [22]:
count = collection.count()

print(f"Total indexed documents: {count}")

Total indexed documents: 8


In [23]:
results = collection.query(
    query_texts=[
        "rear yard requirements in R6 districts"
    ],
    n_results=3
)

results

{'ids': [['zr_03_rear_yard_requirements.md_0',
   'zr_07_front_yard_requirements.md_1',
   'zr_03_rear_yard_requirements.md_2']],
 'embeddings': None,
 'documents': [['[Source: zr_03_rear_yard_requirements.md | Section ID: 23-342 | Amended: 5/12/2021 | Lines: 9-31]\n\n### 23-342: Rear Yard Requirements\n\n**Applicable Districts:** R1 through R10\n\nIn all districts, as indicated, a **rear yard** shall be provided at every required **rear lot line** of a **zoning lot**, except as otherwise provided in Sections 23-341 (Permitted obstructions in required rear yards or rear yard equivalents) and 23-344 (Additional rear yard modifications).\n\nThe minimum required **rear yard** depth shall be:\n\n| District | Minimum Rear Yard Depth |\n|---|---|\n| R1 through R5 | 30 feet |\n| R6 through R10 | 30 feet |\n\nFor corner lots in any Residence District, no rear yard shall be required.\n\nFor through lots in any Residence District, each **street** frontage shall be subject to front yard requireme

# Cross Reference Map Generation

In [24]:
import json

In [25]:
section_map = {}

for chunk in all_chunks:

    metadata = chunk["metadata"]

    section_id = metadata.get(
        "section_id"
    )

    if not section_id:
        continue

    section_map[section_id] = {

        "chunk_id":
        chunk["id"],

        "source_file":
        metadata.get(
            "source_file"
        ),

        "cross_refs":
        metadata.get(
            "cross_refs",
            []
        )
    }

In [26]:
section_map_path = (
    Path(CHROMA_PATH)
    / "section_map.json"
)

with open(
    section_map_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        section_map,
        f,
        indent=2
    )

print(f"Saved: {section_map_path}")


Saved: chroma_db\section_map.json


# Final Stats Display

In [38]:
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()

# =====================================================
# BUILD FILE STATS DIRECTLY FROM ALL_CHUNKS
# =====================================================

file_chunk_counts = {}

file_amendments = {}

for chunk in all_chunks:

    metadata = chunk["metadata"]

    source_file = metadata[
        "source_file"
    ]

    last_amended = metadata[
        "last_amended"
    ]

    if source_file not in file_chunk_counts:

        file_chunk_counts[
            source_file
        ] = 0

    file_chunk_counts[
        source_file
    ] += 1

    file_amendments[
        source_file
    ] = last_amended

# =====================================================
# RICH TABLE
# =====================================================

table = Table(
    title="INTELLI-SITE CORPUS SUMMARY",
    show_lines=True
)

table.add_column(
    "Source File",
    style="cyan",
    overflow="fold"
)

table.add_column(
    "Chunks",
    justify="right",
    style="green"
)

table.add_column(
    "Last Amended",
    style="yellow"
)

total_chunks = 0

for source_file, count in sorted(
    file_chunk_counts.items()
):

    total_chunks += count

    table.add_row(

        source_file,

        str(count),

        str(
            file_amendments[
                source_file
            ]
        )
    )

console.print(table)

# =====================================================
# SUMMARY PANEL
# =====================================================

total_documents = len(
    file_chunk_counts
)

average_chunks = (

    round(
        total_chunks / total_documents,
        2
    )

    if total_documents > 0

    else 0
)

console.print(

    Panel.fit(

        f"""
Total Documents : {total_documents}
Total Chunks    : {total_chunks}
Average Chunks  : {average_chunks}
        """,

        title="Corpus Statistics",

        border_style="bright_blue"
    )
)


                INTELLI-SITE CORPUS SUMMARY                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Source File                      ┃ Chunks ┃ Last Amended ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━┩
│ zr_03_rear_yard_requirements.md  │      3 │ 5/12/2021    │
├──────────────────────────────────┼────────┼──────────────┤
│ zr_07_front_yard_requirements.md │      2 │ 6/3/2020     │
├──────────────────────────────────┼────────┼──────────────┤
│ zr_10_height_setback_R6_R12.md   │      3 │ 4/30/2024    │
└──────────────────────────────────┴────────┴──────────────┘

╭── Corpus Statistics ───╮
│                        │
│ Total Documents : 3    │
│ Total Chunks    : 8    │
│ Average Chunks  : 2.67 │
│                        │
╰────────────────────────╯